In [1]:
#This script trains an extended isolation forest
###
###
###Libraries
set.seed(1234)
options(scipen=999)
library(data.table)
library(reshape2)
library(isotree)
library(GenomicRanges)
library(rtracklayer)
library(dplyr)
library(tidyr)
library(precrec)
library(Rsamtools)
library(ggplot2)
library(ggpattern)
library(gridExtra)
library(universalmotif)
library(TFBSTools)
library(ggseqlogo)
library(patchwork)
library(reshape2)
library(stringr)
library(ggrepel)
library(scales)
library(RcppCNPy)
library(purrr)


Attaching package: ‘reshape2’


The following objects are masked from ‘package:data.table’:

    dcast, melt


Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors


At

In [2]:
###
###
###
###
###
#Functions
###
###
###
###
###

In [2]:
##
##Flips after liftOver
merge_gnomad_flips <- function(df, gnomAD) {
  df$mergeIndex <- seq_len(nrow(df))
  df$id <- paste(df$chr, df$pos, df$ref, df$alt, sep = "_")
  original <- df
  result <- NULL
  for (mode in c("standard", "allele", "strand", "both")) {
    candidates <- original[!(original$mergeIndex %in% result$mergeIndex), , drop = FALSE]
    if (mode %in% c("strand", "both")) {
      candidates$ref <- chartr("ACGT", "TGCA", candidates$ref)
      candidates$alt <- chartr("ACGT", "TGCA", candidates$alt)
    }
    if (mode %in% c("allele", "both")) candidates[c("ref", "alt")] <- candidates[c("alt", "ref")]
    candidates$id <- paste(candidates$chr, candidates$pos, candidates$ref, candidates$alt, sep = "_")
    matched <- merge(candidates, gnomAD, by = "id")
    result <- if (is.null(result)) matched else rbind(result, matched)
    cat(mode, ":", nrow(matched), "matched variants\n")
  }
  result$mergeIndex <- NULL
  return(result)
}

##
##Overlap bed score files
append_interval_score <- function(enf, score_data, score_col = "DVAR", transform = FALSE) {
  colnames(score_data) <- c("chr", "start", "end", "score")
  gr_intervals <- GRanges(seqnames = score_data$chr, ranges = IRanges(start = score_data$start + 1, end = score_data$end), score = score_data$score)
  gr_pos <- GRanges(seqnames = enf$chr, ranges = IRanges(start = enf$pos, end = enf$pos))
  overlaps <- findOverlaps(query = gr_pos, subject = gr_intervals)
  n_overlaps <- tabulate(queryHits(overlaps), nbins = nrow(enf))
  valid <- n_overlaps[queryHits(overlaps)] == 1
  enf[[score_col]] <- NA_real_
  enf[[score_col]][queryHits(overlaps)[valid]] <- mcols(gr_intervals)$score[subjectHits(overlaps)[valid]]
  if (transform) enf[[score_col]] <- -10 * log10(1 - enf[[score_col]])
  enf <- enf[n_overlaps <= 1, , drop = FALSE]
  return(enf)
}

##
##Overlap bed region files
add_overlap <- function(df, regions) {
  gr_regions <- GRanges(seqnames = regions[[1]],ranges = IRanges(start = regions[[2]] + 1, end = regions[[3]]))
  overlapsAny(GRanges(seqnames = df$chr,ranges = IRanges(start = df$pos, end = df$pos)),gr_regions)
}

##
##run isolation forest
runIso <- function(nameMethod,ndimIso,ntreesIso,sampleSizeIso,weighted,datasetTrain,seedIso = 1234) {
        if (weighted == FALSE) {
            trainMod <- isolation.forest(
                datasetTrain[, grep("(^PC)", names(datasetTrain))],
                ndim = ndimIso,
                ntrees = ntreesIso,
                sample_size = sampleSizeIso,
                nthreads = 10,
                seed = seedIso)
        } else {
            trainMod <- isolation.forest(
                datasetTrain[, grep("(^PC)", names(datasetTrain))],
                ndim = ndimIso,
                ntrees = ntreesIso,
                sample_size = sampleSizeIso,
                nthreads = 10,
                seed = seedIso,
                column_weights = c(20:1)
            )
        }
        enf[[nameMethod]] <- predict(trainMod,enf[, grep("(^PC)", names(enf))])
        return(enf)
    }

##
##Parameter optimization
plotAucBar <- function(df, curve_type, bar_color) {
  plot_data <- df %>%
    filter(curvetypes == curve_type) %>%
    arrange(desc(mean_auc)) %>%
    mutate(modnames = factor(modnames,levels = modnames))
  ggplot(plot_data,aes(x = modnames,y = mean_auc)) +
    geom_col(fill = bar_color,width = 0.7) +
    geom_errorbar(
      aes(ymin = mean_auc - sd_auc,ymax = mean_auc + sd_auc),width = 0.2,linewidth = 0.7) +
    labs(x = "Model",y = "AUC") +
    theme_classic() +
    theme(axis.text.x = element_text(angle = 45,hjust = 1))
}

#Tool evaluations
makeEval <- function(caseVar, controlVar, datasetName) {
    s1 <- c(caseVar$eQIF,controlVar$eQIF)
    s2 <- c(caseVar$caddPhred,controlVar$caddPhred)
    s3 <- c(caseVar$LINSIGHT,controlVar$LINSIGHT)
    s4 <- c(caseVar$DVAR,controlVar$DVAR)
    s5 <- c(caseVar$aviPhred,controlVar$aviPhred)
    s6 <- c(caseVar$seqclass_max_absdiff,controlVar$seqclass_max_absdiff)
    scores <- join_scores(s1,s2,s3,s4,s5,s6)
    labels <- c(rep(1, nrow(caseVar)),rep(0, nrow(controlVar)))

    print("The number of cases is:")
    print(nrow(caseVar))
    print("The number of controls is:")
    print(nrow(controlVar))

    model_labels <- c("eQIF","CADD","LINSIGHT","DVAR","AVI","Sei")
    msmdat2 <- mmdata(scores,labels,modnames = model_labels)
    mscurves <- evalmod(msmdat2)
    aucDfShort <- data.frame(auc(mscurves))
    aucDfShort$cohort <- datasetName
    plot_colors <- c("eQIF" = "red","CADD" = "orange","LINSIGHT" = "blue","DVAR" = "green","AVI" = "grey","Sei" = "purple")
    plotROC <- suppressWarnings(autoplot(mscurves,curvetype = "ROC")) +
        geom_line(linewidth = 0.1) +
        scale_color_manual(values = plot_colors) +
        labs(title = NULL,subtitle = NULL) +
        theme(legend.position = "none")
    plotPR <- suppressWarnings(autoplot(mscurves,curvetype = "PR")) +
        geom_line(linewidth = 0.1) +
        scale_color_manual(values = plot_colors) +
        labs(title = NULL,subtitle = NULL) +
        theme(aspect.ratio = NULL,
            axis.title = element_text(size = 8),
            axis.text = element_text(size = 6),
            legend.title = element_blank(),
            legend.text = element_text(size = 6),
            legend.key.width = unit(0.5, "cm"),
            legend.key.height = unit(0.3, "cm"),
            legend.spacing.x = unit(0.2, "cm"),
            legend.spacing.y = unit(0.1, "cm"))
    scores_df <- as.data.frame(scores)
    rocs <- lapply(seq_along(model_labels),
        function(i) {pROC::roc(response = labels,predictor = scores_df[[i]],direction = "<",quiet = TRUE)})
    names(rocs) <- model_labels
    model_pairs <- combn(seq_along(model_labels),2)
    pairwise_roc_results <- vector("list",ncol(model_pairs))
    for (i in seq_len(ncol(model_pairs))) {
        m1 <- model_pairs[1, i]
        m2 <- model_pairs[2, i]
        roc1 <- rocs[[m1]]
        roc2 <- rocs[[m2]]
        test <- pROC::roc.test(roc1,roc2,paired = TRUE,method = "delong",conf.level = 0.95)
        ci1 <- pROC::ci.auc(roc1,conf.level = 0.95)
        ci2 <- pROC::ci.auc(roc2,conf.level = 0.95)
        auc1 <- as.numeric(pROC::auc(roc1))
        auc2 <- as.numeric(pROC::auc(roc2))
        pairwise_roc_results[[i]] <- data.frame(
            cohort = datasetName,
            model1 = model_labels[m1],
            model2 = model_labels[m2],
            auc_model1 = auc1,
            auc_model1_ci_lower =
                as.numeric(ci1[1]),
            auc_model1_ci_upper =
                as.numeric(ci1[3]),
            auc_model2 = auc2,
            auc_model2_ci_lower =
                as.numeric(ci2[1]),
            auc_model2_ci_upper =
                as.numeric(ci2[3]),
            auc_difference =
                auc1 - auc2,
            auc_difference_ci_lower =
                as.numeric(test$conf.int[1]),
            auc_difference_ci_upper =
                as.numeric(test$conf.int[2]),
            p_value =
                test$p.value,
            stringsAsFactors = FALSE
        )
    }
    pairwise_roc_df <- do.call(rbind,pairwise_roc_results)
    rownames(pairwise_roc_df) <- NULL
    pairwise_roc_df$p_value_adj <- p.adjust(pairwise_roc_df$p_value,method = "BH")
    pairwise_roc_df <- pairwise_roc_df[
        order(pairwise_roc_df$p_value),
        ,
        drop = FALSE
    ]
    return(
        list(plotROC = plotROC,plotPR = plotPR,aucDfShort = aucDfShort,deLong = pairwise_roc_df)
    )
}

##
##Locus evaluation
plot_local_manhattan <- function(locusDf, highlight_rsid,
                                 output_prefix = NULL,
                                 width = 12, height = 6, dpi = 600) {
locusDf <- locusDf %>%mutate(CADD = rank(caddPhred),LINSIGHT = rank(LINSIGHT),DVAR = rank(DVAR),eQIF = rank(eQIF), AVI = rank(aviPhred), Sei = rank(seqclass_max_absdiff))
locusDf <- locusDf %>% pivot_longer(cols = c(CADD, LINSIGHT, DVAR, eQIF, AVI, Sei),names_to = "score_type",values_to = "score")
  locusDf$score_type <- factor(locusDf$score_type,levels = c("eQIF", "CADD", "LINSIGHT", "DVAR", "AVI", "Sei"))
  offsets <- c(eQIF = -70,CADD = 0,LINSIGHT = 15,DVAR = 70,AVI = 25,Sei = -25)
  locusDf <- locusDf %>%
    mutate(xplot = pos + offsets[as.character(score_type)])
  lead <- subset(locusDf, rsID == highlight_rsid)
  rect.df <- locusDf %>%
    group_by(rsID, pos) %>%
    summarise(
      ymax = max(score) + 0.35,
      .groups = "drop"
    ) %>%
    mutate(
      xmin = pos - 80,
      xmax = pos + 80,
      ymin = 0.5,
      causal = rsID == highlight_rsid
    )
  plotLocalManhattan <- ggplot(locusDf,aes(x = xplot, y = score)) +
    geom_rect(
      data = subset(rect.df, !causal),
      aes(xmin = xmin,xmax = xmax,ymin = ymin,ymax = ymax),
      inherit.aes = FALSE,
      fill = NA,
      colour = "grey60",
      linewidth = 0.6,
      linetype = 2) +
    geom_rect(data = subset(rect.df, causal),aes(xmin = xmin,xmax = xmax,ymin = ymin,ymax = ymax),
      inherit.aes = FALSE,
      fill = NA,
      colour = "red",
      linewidth = 1.2) +
    geom_text(data = rect.df,aes(x = pos,y = ymax + 0.15,label = rsID),
      inherit.aes = FALSE,
      angle = 90,
      hjust = 0,
      vjust = 0.5,
      size = 2.5) +
    geom_point(aes(shape = score_type,colour = score_type),size = 3) +
    scale_colour_manual(
      name = "Score",
      values = c(
        eQIF = "red",
        CADD = "orange",
        LINSIGHT = "blue",
        DVAR = "green",
        AVI = "grey",
        Sei = "purple")) +
    scale_x_continuous(labels = function(x) sprintf("%.3f", x / 1e6),expand = expansion(mult = c(0.02, 0.08))) +
    scale_y_continuous(
      breaks = 1:(nrow(locusDf) / 6),
      labels = c((nrow(locusDf) / 6):1),
      limits = c(0, (nrow(locusDf) / 6) + 1.5),
      expand = expansion(mult = c(0, 0))) +
    labs(x = "Genomic position (Mb, hg38)",y = "Rank",shape = "Score",colour = "Score") +
    theme_classic(base_size = 18) +
    theme(axis.title = element_text(),legend.title = element_text(),legend.position = "right")
    ggsave(paste0(output_prefix, ".pdf"),plot = plotLocalManhattan,width = width,height = height)
    ggsave(paste0(output_prefix, ".png"),plot = plotLocalManhattan,width = width,height = height,dpi = dpi)
  return(plotLocalManhattan)
}

##
##Make TFBS count plots
plotMotifMutations <- function(enf,motif_path,motif_length,flank = 16,top_n = 5000,regions_file = "../data/outputs/eval/fimo/regions.txt",
    fimo_file = "../data/outputs/eval/fimo/fimo.tsv",fimo_script = "./fimo.sh") {
  enfTop <- enf[order(-enf$eQIF, enf$id), ]
  enfTop <- enfTop[1:top_n, ]
  regions <- paste0(enfTop$chr, ":",enfTop$pos - flank, "-",enfTop$pos + flank)
  write.table(regions,file = regions_file,col.names = FALSE,row.names = FALSE,quote = FALSE)
  system(sprintf("%s %s",shQuote(fimo_script),shQuote(motif_path)))
  fimoDf <- read.table(fimo_file,header = TRUE,sep = "\t")
  snps_gr <- GRanges(seqnames = enfTop$chr,ranges = IRanges(start = enfTop$pos, end = enfTop$pos))
  motifs_gr <- GRanges(seqnames = fimoDf$sequence_name,ranges = IRanges(start = fimoDf$start, end = fimoDf$stop),strand = fimoDf$strand)
  hits <- findOverlaps(snps_gr, motifs_gr)
  overlap_df <- data.frame(snp_pos = start(snps_gr)[queryHits(hits)],
    motif_start = start(motifs_gr)[subjectHits(hits)],
    motif_end = end(motifs_gr)[subjectHits(hits)],
    strand = as.character(strand(motifs_gr)[subjectHits(hits)]))
  overlap_df <- overlap_df %>%
    mutate(relative_pos = ifelse(strand == "+",snp_pos - motif_start + 1,motif_end - snp_pos + 1)) %>% filter(relative_pos >= 1 & relative_pos <= motif_length)
  mutation_counts <- overlap_df %>% count(relative_pos, name = "mutation_count") %>% complete(relative_pos = 1:motif_length,fill = list(mutation_count = 0))
  motif <- read_meme(motif_path)
  pfm_matrix <- motif["motif"]
  plot_top <- ggseqlogo(pfm_matrix, method = "bits") +
    theme_minimal() + theme(axis.text.x = element_blank(),axis.ticks.x = element_blank()) + labs(x = "")
  plot_bottom <- ggplot(mutation_counts,aes(relative_pos, mutation_count)) +geom_col(fill = "firebrick", width = 0.7) +scale_x_continuous(
      breaks = 1:motif_length,limits = c(0.5, motif_length + 0.5)) +scale_y_continuous(labels = abs) +labs(x = "Position in motif",y = "Mutation count") +
    theme_minimal()
  combined_plot <- plot_top / plot_bottom
  print(cor.test(apply(pfm_matrix, 2, max), mutation_counts$mutation_count, method = "spearman"))
  return(combined_plot)
}

In [4]:
###
###
###
###
###
#Preprocessing
###
###
###
###
###

In [5]:
##
##Read gnomAD
gnomAD <- fread("../data/outputs/gnomAD/gn_v3_red_inf_id2_IDs.list", data.table = FALSE, header = FALSE, select = c(1))
colnames(gnomAD) <- "id"

In [6]:
##
##Annotate Sei
sei <- do.call(rbind, lapply(1:22, function(chr) {
  print(paste("Read chromosome", chr))
  npyLoad(sprintf("../data/outputs/Sei/gn_v3_red_inf_id2_seiAll_chr%d.sequence_class_scores.npy",chr))}))
sei <- as.data.frame(sei)
seqclass_max_absdiff <- apply(sei, 1, function(x) max(abs(x)))

[1] "Read chromosome 1"
[1] "Read chromosome 2"
[1] "Read chromosome 3"
[1] "Read chromosome 4"
[1] "Read chromosome 5"
[1] "Read chromosome 6"
[1] "Read chromosome 7"
[1] "Read chromosome 8"
[1] "Read chromosome 9"
[1] "Read chromosome 10"
[1] "Read chromosome 11"
[1] "Read chromosome 12"
[1] "Read chromosome 13"
[1] "Read chromosome 14"
[1] "Read chromosome 15"
[1] "Read chromosome 16"
[1] "Read chromosome 17"
[1] "Read chromosome 18"
[1] "Read chromosome 19"
[1] "Read chromosome 20"
[1] "Read chromosome 21"
[1] "Read chromosome 22"


In [7]:
gnomAD <- cbind(gnomAD, seqclass_max_absdiff)

In [8]:
##
##Read Enformer
enf <- fread("../data/outputs/enf/enfHg38.txt", data.table = FALSE, header = TRUE)
enf <- enf[complete.cases(enf),]
enf$chr <- paste0("chr", enf$chr)
enf <- enf[!duplicated(enf[, 1:4]) & 
                !duplicated(enf[, 1:4], fromLast = TRUE), ]
enf$id <- paste0(enf$chr, "_", enf$pos, "_", enf$ref, "_", enf$alt)

In [9]:
##
##subset by gnomAD pos
gnomADPosDf <- cbind(
  gnomAD$id,
  colsplit(gnomAD$id, "_", names = c("chr", "pos", "ref", "alt"))
)

enf <- enf[enf$pos %in% unique(gnomADPosDf$pos),]

In [10]:
##
##Consider flips for enf
#Merge standard
enf <- merge_gnomad_flips(enf, gnomAD)

standard : 7837613 matched variants
allele : 1580956 matched variants
strand : 11196 matched variants
both : 2204 matched variants


In [11]:
##
## Read and annotate conservation metrics
bwFiles <- c(phyloPVertebrates = "../data/raw/phylos/hg38.phyloP100way.bw",
             phyloP447 = "../data/raw/phylos/hg38.phyloP447way.bw",
             phyloP447Primates = "../data/raw/phylos/hg38.phyloP447wayPrimates.bw")

labels <- c(phyloPVertebrates = "vert", phyloP447 = "mammal", phyloP447Primates = "primate")
window_size <- 1000000L
chrs <- as.character(enf$chr)
pos <- enf$pos

for (score in names(bwFiles)) {
  bw <- BigWigFile(bwFiles[[score]])
  chr_lengths <- seqlengths(seqinfo(bw))
  valid_chrs <- intersect(unique(chrs[!is.na(chrs)]), names(chr_lengths))
  values <- rep(NA_real_, nrow(enf))

  for (chr in valid_chrs) {
    chr_length <- chr_lengths[[chr]]
    if (is.na(chr_length)) next
    idx <- which(chrs == chr & !is.na(pos) & pos >= 1 & pos <= chr_length)
    if (length(idx) == 0L) next
    windows <- split(idx, (pos[idx] - 1) %/% window_size)

    for (i in seq_along(windows)) {
      ix <- windows[[i]]
      win_start <- floor((min(pos[ix]) - 1) / window_size) * window_size + 1
      win_end <- min(win_start + window_size - 1, chr_length)
      region <- GRanges(seqnames = chr, ranges = IRanges(start = win_start, end = win_end))
      bw_gr <- import(bw, which = region, as = "GRanges")

      if (length(bw_gr) > 0L) {
        query_gr <- GRanges(seqnames = chr, ranges = IRanges(start = pos[ix], width = 1L))
        hits <- findOverlaps(query_gr, bw_gr, ignore.strand = TRUE)
        if (length(hits) > 0L) {
          values[ix[queryHits(hits)]] <- as.numeric(mcols(bw_gr)$score[subjectHits(hits)])
        }
      }
    }
  }

  enf[[score]] <- values
  cat(labels[[score]], "done\n")
  flush.console()
}

vert done
mammal done
primate done


In [12]:
##
##Read and annotate CADD, and AVI
#CADD
runFirst <- FALSE
enfPos <- unique(enf[,c("chr","pos")])
enfPosBed <- cbind(enfPos, enfPos[,2])
enfPosBed[,2] <- enfPosBed[,2] -1
enfPosBed[,1] <- gsub("chr","",enfPosBed[,1])

if(runFirst == TRUE){
    fwrite(
      enfPosBed,
      "../data/outputs/cadd/query.bed",
      sep = "\t",
      col.names = FALSE
    )
    system("tabix -R ../data/outputs/cadd/query.bed ../data/raw/cadd/whole_genome_SNVs_38.tsv.gz > ../data/outputs/cadd/cadd_hits.tsv")
}
cadd <- fread("../data/outputs/cadd/cadd_hits.tsv", header = FALSE, data.table = FALSE, select = c(1,2,3,4,6))
cadd$V1 <- paste0("chr", cadd$V1)
colnames(cadd) <- c("chr","pos","ref","alt","caddPhred")
enf <- merge(enf, cadd, by = c("chr","pos","ref","alt"), all.x = TRUE)

if(runFirst == TRUE){
    enfPosBed[,1] <- paste0("chr",enfPosBed[,1])
    fwrite(
      enfPosBed,
      "../data/outputs/avi/query.bed",
      sep = "\t",
      col.names = FALSE
    )
    system("tabix -R ../data/outputs/avi/query.bed ../data/raw/avi/alphagenome_variant_impact_score_snvs.tsv.gz > ../data/outputs/avi/avi_hits.tsv")
}

avi <- fread("../data/outputs/avi/avi_hits.tsv", header = FALSE, data.table = FALSE, select = c(1,2,3,4,6))
colnames(avi) <- c("chr","pos","ref","alt","aviPhred")
enf <- merge(enf, avi, by = c("chr","pos","ref","alt"), all.x = TRUE)

In [13]:
##
##DVAR prep
if(runFirst == TRUE){
    system2(
      "bedtools",
      args = c(
        "intersect", "-u",
        "-a", "../data/outputs/dvar/hg38_DVAR.score.sorted.merged.bed",
        "-b", "../data/outputs/avi/query.bed"
      ),
      stdout = "../data/outputs/dvar/dvarCommon.bed"
    )
}

In [14]:
##
##Annotate DVAR and LINSIGHT
DVAR <- fread("../data/outputs/dvar/dvarCommon.bed", header = FALSE, data.table = FALSE)
LINSIGHT <- fread("../data/outputs/linsight/LINSIGHT_HG38_sorted.bed", header = FALSE, data.table = FALSE)
enf <- append_interval_score(enf, DVAR, score_col = "DVAR", transform = FALSE)
enf <- append_interval_score(enf, LINSIGHT, score_col = "LINSIGHT", transform = FALSE)

In [15]:
##
##Median imputation
cols <- grep("^(PC|phyloP|caddPhred|aviPhred|LINSIGHT|DVAR|seqclass_max_absdiff)", names(enf), value = TRUE)
for (col in cols) {
  med <- median(enf[[col]], na.rm = TRUE)
  enf[[col]][is.na(enf[[col]])] <- med
  enf[[col]][is.infinite(enf[[col]])] <- med
}

In [16]:
##
##Read eQTLvariants for comparison
eQTLVarCase <- fread("../data/outputs/gtexEqtl/eqtlCaseIds.txt", header = FALSE, data.table = FALSE)
eQTLVarControl <- fread("../data/outputs/gtexEqtl/eqtlControlIds.txt", header = FALSE, data.table = FALSE)

In [17]:
###
###
###
###
###
#Parameter optimization
###
###
###
###
###

In [18]:
enf <- enf[order(enf$id), , drop = FALSE]
rownames(enf) <- NULL

In [19]:
allAucDf <- list()
for (step in 1:5) {
    set.seed(1234 + step)
    idx <- sample(seq_len(nrow(enf)),size = floor(0.8 * nrow(enf)))
    enfTrain <- enf[idx, ]
    enf <- runIso("eQIF_IF_500_1000",1, 500, 1000, FALSE,enfTrain,1234 + step)
    enf <- runIso("eQIF_IF_500_10000",1, 500, 10000, FALSE,enfTrain,1234 + step)
    enf <- runIso("eQIF_IF_500_1000_w",1, 500, 1000, TRUE,enfTrain,1234 + step)
    enf <- runIso("eQIF_IF_500_10000_w",1, 500, 10000, TRUE,enfTrain,1234 + step)
    enf <- runIso("eQIF_EIF_500_1000",2, 500, 1000, FALSE,enfTrain,1234 + step)
    enf <- runIso("eQIF_EIF_500_10000",2, 500, 10000, FALSE,enfTrain,1234 + step)
    enf <- runIso("eQIF_EIF_500_1000_w",2, 500, 1000, TRUE,enfTrain,1234 + step)
    enf <- runIso("eQIF_EIF_500_10000_w",2, 500, 10000, TRUE,enfTrain,1234 + step)
    enfTest <- enf[-idx, ]
    eQTLCaseVar <- enfTest[enfTest$id %in% eQTLVarCase$V1,]
    eQTLControlVar <- enfTest[enfTest$id %in% eQTLVarControl$V1,]
    s1 <- c(eQTLCaseVar$eQIF_IF_500_1000,eQTLControlVar$eQIF_IF_500_1000)
    s2 <- c(eQTLCaseVar$eQIF_IF_500_10000,eQTLControlVar$eQIF_IF_500_10000)
    s3 <- c(eQTLCaseVar$eQIF_IF_500_1000_w,eQTLControlVar$eQIF_IF_500_1000_w)
    s4 <- c(eQTLCaseVar$eQIF_IF_500_10000_w,eQTLControlVar$eQIF_IF_500_10000_w)
    s5 <- c(eQTLCaseVar$eQIF_EIF_500_1000,eQTLControlVar$eQIF_EIF_500_1000)
    s6 <- c(eQTLCaseVar$eQIF_EIF_500_10000,eQTLControlVar$eQIF_EIF_500_10000)
    s7 <- c(eQTLCaseVar$eQIF_EIF_500_1000_w,eQTLControlVar$eQIF_EIF_500_1000_w)
    s8 <- c(eQTLCaseVar$eQIF_EIF_500_10000_w,eQTLControlVar$eQIF_EIF_500_10000_w)
    scores <- join_scores(s1, s2, s3, s4,s5, s6, s7, s8)
    labels <- c(rep(1, nrow(eQTLCaseVar)),rep(0, nrow(eQTLControlVar)))
    model_labels <- c("IF-500-1000","IF-500-10000","IF-500-1000-w","IF-500-10000-w","EIF-500-1000","EIF-500-10000","EIF-500-1000-w","EIF-500-10000-w")
    msmdat2 <- mmdata(scores,labels,modnames = model_labels)
    mscurves <- evalmod(msmdat2)
    aucDf_step <- data.frame(auc(mscurves))
    aucDf_step$cohort <- paste0("parameter optimization step ",step)
    aucDf_step$step <- step
    allAucDf[[step]] <- aucDf_step
    cat("Number of cases:", nrow(eQTLCaseVar), "\n")
    cat("Number of controls:", nrow(eQTLControlVar), "\n")
    cat("Finished step:", step, "\n")
}

aucDf <- do.call(rbind,allAucDf)
rownames(aucDf) <- NULL

Number of cases: 2536 
Number of controls: 153401 
Finished step: 1 
Number of cases: 2499 
Number of controls: 153013 
Finished step: 2 
Number of cases: 2527 
Number of controls: 153823 
Finished step: 3 
Number of cases: 2475 
Number of controls: 153658 
Finished step: 4 
Number of cases: 2523 
Number of controls: 153512 
Finished step: 5 


In [20]:
##
##AUC summary
aucMetParamOpt <- aucDf %>%
  group_by(curvetypes, modnames) %>%
  summarise(
    mean_auc = mean(aucs, na.rm = TRUE),
    sd_auc = sd(aucs, na.rm = TRUE),
    n = sum(!is.na(aucs)),
    .groups = "drop")

plotAuRocBar <- plotAucBar(aucMetParamOpt,"ROC","steelblue")
plotAuPrcBar <- plotAucBar(aucMetParamOpt,"PRC","darkorange")
plotParam <- (plotAuRocBar | plotAuPrcBar) + 
  plot_annotation(tag_levels = "A", tag_suffix = ")",theme = theme(plot.tag = element_text(size = 24, face = "bold"))) +
  plot_layout()

ggsave("../data/outputs/eval/paramTune.pdf", plot = plotParam, width = 7.5, height = 3.5)
ggsave("../data/outputs/eval/paramTune.png", plot = plotParam, width = 7.5, height = 3.5, dpi = 600)

In [21]:
##
##Make final eQIF score
enf <- enf[order(enf$id), , drop = FALSE]
enf <- enf[, !grepl("^eQIF", names(enf))]
enf <- runIso("eQIF",1, 500, 1000, TRUE,enf)

In [22]:
###
###
###
###
###
#Variant analysis
###
###
###
###
###

In [23]:
##
##Get score correlations
eQTLCaseVar <- enf[enf$id %in% eQTLVarCase$V1,]
eQTLControlVar <- enf[enf$id %in% eQTLVarControl$V1,]
caseCon <- rbind(eQTLCaseVar, eQTLControlVar)
enfCor <- caseCon[, c("caddPhred","aviPhred", "LINSIGHT", "DVAR", "eQIF", "seqclass_max_absdiff")]
vars <- c("eQIF","caddPhred","aviPhred","LINSIGHT","seqclass_max_absdiff","DVAR")

enfCor <- caseCon[, vars]
cor_matrix <- cor(enfCor,method = "spearman",use = "complete.obs")
cor_matrix <- cor_matrix[vars, vars]

cor_df <- melt(cor_matrix)
plotHeat <- ggplot(cor_df, aes(Var1, Var2, fill = value)) +
  geom_tile() + geom_text(aes(label = round(value, 2)), color = "black", size = 1.5) +
  scale_fill_gradient2(low = "blue",
                       mid = "white",
                       high = "red",
                       midpoint = 0,
                       limits = c(-1, 1)) +
  coord_fixed() +
  theme_minimal() +
  labs(fill = "CC", x = NULL,y = NULL) +
  theme(axis.text.x = element_text(size = 6, angle = 45, hjust = 1),
    axis.text.y = element_text(size = 6),
    legend.title = element_blank(),
    legend.text = element_text(size = 6)) + scale_x_discrete(labels = c(caddPhred = "CADD", aviPhred = "AVI", seqclass_max_absdiff = "Sei"))+
    scale_y_discrete(labels = c(caddPhred = "CADD", aviPhred = "AVI", seqclass_max_absdiff = "Sei"))

ggsave("../data/outputs/eval/heatPlot.pdf", plot = plotHeat, width = 7.5, height = 3.5)
ggsave("../data/outputs/eval/heatPlot.png", plot = plotHeat, width = 7.5, height = 3.5, dpi = 600)

In [24]:
##
##Add overlap with exons & gene regulatory regions
coding <- fread("../data/outputs/reg/gencode.v44.cds.bed", header = FALSE, data.table = FALSE)
enf$overlap_coding <- add_overlap(enf, coding)

regReg <- fread("../data/raw/reg/GRCh38-cCREs.bed", header = FALSE, data.table = FALSE)
fp <- fread("../data/raw/reg/consensus_footprints_and_collapsed_motifs_hg38.bed", header = FALSE, data.table = FALSE)
enf$regReg <- add_overlap(enf, regReg)
enf$prom <- add_overlap(enf, regReg[grepl(pattern = "PLS", x = regReg$V6),])
enf$enh <- add_overlap(enf, regReg[grepl(pattern = "ELS", x = regReg$V6),])
enf$fp <- add_overlap(enf, fp)

In [25]:
##
##Compare with other scores for global eQTL prediction
resCompRoc_eQTL_classicComp_allVar <- makeEval(caseVar = eQTLCaseVar,controlVar = eQTLControlVar,datasetName = "eQTL cohort")
aucDf <- resCompRoc_eQTL_classicComp_allVar[[3]]

# DeLong results
deLong_eQTL_allVar <- resCompRoc_eQTL_classicComp_allVar[[4]]
pROC1 <- resCompRoc_eQTL_classicComp_allVar[[1]]
pPR1  <- resCompRoc_eQTL_classicComp_allVar[[2]]
pROC1 <- pROC1 +
    theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
pPR1 <- pPR1 +
    theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
ploteQTL <- (pROC1 | pPR1) +plot_layout(guides = "collect") &theme(legend.position = "none")
ploteQTL <- ploteQTL +
    plot_annotation(tag_levels = "A",tag_suffix = ")",theme = theme(plot.tag = element_text(size = 24,face = "bold")))

#Coding eQTLs
eQTLCaseVarCoding <- enf[enf$id %in% eQTLVarCase$V1 &enf$overlap_coding == TRUE,]
eQTLControlVarCoding <- enf[enf$id %in% eQTLVarControl$V1 &enf$overlap_coding == TRUE,]
resCompRoc_eQTL_classicComp_codingVar <- makeEval(caseVar = eQTLCaseVarCoding,controlVar = eQTLControlVarCoding,datasetName = "eQTL coding cohort")
aucDf <- rbind(aucDf,resCompRoc_eQTL_classicComp_codingVar[[3]])

#DeLong results
deLong_eQTL_coding <- resCompRoc_eQTL_classicComp_codingVar[[4]]
pROC1 <- resCompRoc_eQTL_classicComp_codingVar[[1]]
pPR1  <- resCompRoc_eQTL_classicComp_codingVar[[2]]
pROC1 <- pROC1 +
    theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
pPR1 <- pPR1 + theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
ploteQTLCod <- (pROC1 | pPR1) +
    plot_layout(guides = "collect") &
    theme(
        legend.position = "bottom",
        legend.justification = "center",
        legend.title = element_blank(),
        legend.text = element_text(size = 10),
        legend.key.width = unit(0.5, "cm"),
        legend.key.height = unit(0.3, "cm"),
        legend.spacing.x = unit(0.1, "cm"),
        legend.spacing.y = unit(0.1, "cm"),
        legend.margin = margin(0, 0, 0, 0),
        legend.box.margin = margin(0, 0, 0, 0)
    )

ploteQTLCod <- ploteQTLCod +
    plot_annotation(
        tag_levels = "A",
        tag_suffix = ")",
        theme = theme(
            plot.tag = element_text(
                size = 24,
                face = "bold"
            )
        )
    )

#Save DeLong results
deLong_eQTL_all <- rbind(deLong_eQTL_allVar,deLong_eQTL_coding)
write.table(deLong_eQTL_all,file = "../data/outputs/eval/eQTL_classicComp_deLong.txt",append = FALSE,quote = FALSE,row.names = FALSE,col.names = TRUE,sep = "\t")

#Save Plots
ploteQTLAll <- (ploteQTL / ploteQTLCod) +
    plot_annotation(tag_levels = "A",tag_suffix = ")",theme = theme(plot.tag = element_text(size = 24,face = "bold"))) +
    plot_layout()
ggsave("../data/outputs/eval/eQTL_classicComp.pdf",plot = ploteQTLAll,width = 7.5,height = 5)
ggsave("../data/outputs/eval/eQTL_classicComp.png",plot = ploteQTLAll,width = 7.5,height = 5,dpi = 600)

[1] "The number of cases is:"
[1] 12727
[1] "The number of controls is:"
[1] 767512
[1] "The number of cases is:"
[1] 553
[1] "The number of controls is:"
[1] 6244


In [26]:
##
##Read GWAS variants for comparison
gwasVarCase <- fread("../data/outputs/gtexEqtl/eqtlGwasCaseIds.txt", header = FALSE, data.table = FALSE)
gwasVarControl <- fread("../data/outputs/gtexEqtl/eqtlGwasControlIds.txt", header = FALSE, data.table = FALSE)

In [27]:
##
##Compare with other scores for GWAS prediction
gwasCaseVar <- enf[enf$id %in% gwasVarCase$V1,]
gwasControlVar <- enf[enf$id %in% gwasVarControl$V1,]
resComp_GWASeQTL_classicComp_allVar <- makeEval(caseVar = gwasCaseVar,
    controlVar = gwasControlVar,
    datasetName = "GWASeQTL cohort")

aucDf <- rbind(aucDf,resComp_GWASeQTL_classicComp_allVar[[3]])
deLong_GWASeQTL_allVar <- resComp_GWASeQTL_classicComp_allVar[[4]]
pROC1 <- resComp_GWASeQTL_classicComp_allVar[[1]]
pPR1  <- resComp_GWASeQTL_classicComp_allVar[[2]]
pROC1 <- pROC1 +
    theme(axis.title.x = element_text(size = 8),
        axis.title.y = element_text(size = 8),
        axis.text.x  = element_text(size = 6),
        axis.text.y  = element_text(size = 6))
pPR1 <- pPR1 +
    theme(
        axis.title.x = element_text(size = 8),
        axis.title.y = element_text(size = 8),
        axis.text.x  = element_text(size = 6),
        axis.text.y  = element_text(size = 6)
    )
ploteGWAS <- (pROC1 | pPR1) +
    plot_layout(guides = "collect") &
    theme(legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 10),
        legend.key.width = unit(0.5, "cm"),
        legend.key.height = unit(0.3, "cm"),
        legend.spacing.x = unit(0.1, "cm"),
        legend.spacing.y = unit(0.1, "cm"),
        legend.margin = margin(0, 0, 0, 0),
        legend.box.margin = margin(0, 0, 0, 0)
    )
ploteGWAS <- ploteGWAS +
    plot_annotation(
        tag_levels = "A",
        tag_suffix = ")",
        theme = theme(
            plot.tag = element_text(
                size = 24,
                face = "bold"
            )
        )
    )
write.table(deLong_GWASeQTL_allVar,file = "../data/outputs/eval/GWASeQTL_classicComp_deLong.txt",append = FALSE,quote = FALSE,row.names = FALSE,col.names = TRUE,sep = "\t")
ggsave("../data/outputs/eval/GWASeQTL_classicComp_allVar.pdf",plot = ploteGWAS,width = 7.5,height = 4)
ggsave("../data/outputs/eval/GWASeQTL_classicComp_allVar.png",plot = ploteGWAS,width = 7.5,height = 4,dpi = 600)

[1] "The number of cases is:"
[1] 357
[1] "The number of controls is:"
[1] 27394


In [28]:
##
##Evaluate different models on MPRA data
chain <- import.chain("../data/raw/chain/hg19ToHg38.over.chain")
mpraAll <- fread("../data/raw/mpra/GTExMpra.txt", data.table = FALSE, header = TRUE)
mpraAll <- makeGRangesFromDataFrame(mpraAll, seqnames.field = "chr", start.field = "pos", end.field = "pos", keep.extra.columns = TRUE) |> liftOver(chain)
mpraAll <- unlist(mpraAll[elementNROWS(mpraAll) == 1])
mpraAll <- as.data.frame(mpraAll)
mpraAll <- mpraAll[, !names(mpraAll) %in% c("end", "width", "strand")]
names(mpraAll)[1:2] <- c("chr", "pos")
mpraAll <- merge_gnomad_flips(mpraAll, gnomAD)
mpraAll$id <- paste0(mpraAll$chr, "_", mpraAll$pos, "_", mpraAll$ref, "_", mpraAll$alt)

MPRACaseVar <- enf[enf$id %in% mpraAll$id[mpraAll$emVar == TRUE],]
MPRAControlVar <- enf[enf$id %in% mpraAll$id[mpraAll$emVar == FALSE],]
resCompRoc_MPRA_classicComp_allVar <- makeEval(caseVar = MPRACaseVar,controlVar = MPRAControlVar,datasetName = "MPRA cohort")
aucDf <- rbind(aucDf,resCompRoc_MPRA_classicComp_allVar[[3]])

# DeLong results
deLong_MPRA_allVar <- resCompRoc_MPRA_classicComp_allVar[[4]]
pROC1 <- resCompRoc_MPRA_classicComp_allVar[[1]]
pPR1  <- resCompRoc_MPRA_classicComp_allVar[[2]]
pROC1 <- pROC1 +
    theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
pPR1 <- pPR1 +
    theme(axis.title.x = element_text(size = 8),axis.title.y = element_text(size = 8),axis.text.x  = element_text(size = 6),axis.text.y  = element_text(size = 6))
plotMPRA <- (pROC1 | pPR1) +plot_layout(guides = "collect")
plotMPRA <- plotMPRA +
    plot_annotation(tag_levels = "A",tag_suffix = ")",theme = theme(plot.tag = element_text(size = 24,face = "bold")))
write.table(deLong_MPRA_allVar,file = "../data/outputs/eval/MPRA_deLong.txt",append = FALSE,quote = FALSE,row.names = FALSE,col.names = TRUE,sep = "\t")
ggsave("../data/outputs/eval/MPRA.pdf",plot = plotMPRA,width = 7.5,height = 4)
ggsave("../data/outputs/eval/MPRA.png",plot = plotMPRA,width = 7.5,height = 4,dpi = 600)

standard : 263560 matched variants
allele : 1570 matched variants
strand : 511 matched variants
both : 51 matched variants
[1] "The number of cases is:"
[1] 32212
[1] "The number of controls is:"
[1] 229559


In [29]:
##
##Write aucDf to output
aucDf <- aucDf %>%
  pivot_wider(
    names_from = curvetypes,
    values_from = aucs,
    names_prefix = "auc_")
write.table(aucDf,file = paste0("../data/outputs/eval/aucs.txt"),append = FALSE,quote = FALSE,row.names = FALSE, col.names = TRUE, sep = "\t")

In [30]:
##
##MX2 locus liftover
mx2 <- fread("../data/raw/candidateLoci/mx2.txt", data.table = FALSE)
mx2 <- makeGRangesFromDataFrame(mx2, seqnames.field = "chr", start.field = "pos", end.field = "pos", keep.extra.columns = TRUE) |> liftOver(chain)
mx2 <- unlist(mx2[elementNROWS(mx2) == 1])
mx2 <- as.data.frame(mx2)
mx2 <- mx2[, !names(mx2) %in% c("end", "width", "strand")]
names(mx2)[1:2] <- c("chr", "pos")
mx2$id <- with(mx2, paste(chr, pos, ref, alt, sep = "_"))
mx2 <- merge(mx2, enf)

In [31]:
##
##Read SORT1
sort1 <- fread("../data/raw/candidateLoci/sort1.txt", data.table = FALSE, header = TRUE)
sort1 <- merge(sort1, enf)

In [32]:
##
##Plot loci
sort1Plot <- plot_local_manhattan(locusDf = sort1, highlight_rsid = "rs12740374", output_prefix = "../data/outputs/eval/SORT1_localManhattan", width = 12, height = 6, dpi = 600)
mx2Plot <- plot_local_manhattan(locusDf = mx2, highlight_rsid = "rs398206", output_prefix = "../data/outputs/eval/MX2_localManhattan", width = 12, height = 6, dpi = 600)
plotLoci <- (sort1Plot / mx2Plot) +
  plot_annotation(tag_levels = "A",tag_suffix = ")",theme = theme(plot.tag = element_text(size = 24,face = "bold"))) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom")
ggsave("../data/outputs/eval/loci.pdf",plot = plotLoci,width = 10,height = 10)
ggsave("../data/outputs/eval/loci.png",plot = plotLoci,width = 10,height = 10,dpi = 600)

In [33]:
##
##Make top scores
enfShort <- enf[,colnames(enf) %in% c("chr","pos","ref","alt", "eQIF","caddPhred", "LINSIGHT","DVAR","aviPhred", "seqclass_max_absdiff")]
enfShort$eQIF_quant <- 1 - ((rank(enfShort$eQIF, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$eQIF)) - 1))
enfShort$caddQuant <- 1 - ((rank(enfShort$caddPhred, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$caddPhred)) - 1))
enfShort$aviQuant <- 1 - ((rank(enfShort$aviPhred, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$aviPhred)) - 1))
enfShort$linsightQuant <- 1 - ((rank(enfShort$LINSIGHT, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$LINSIGHT)) - 1))
enfShort$dvarQuant <- 1 - ((rank(enfShort$DVAR, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$DVAR)) - 1))
enfShort$seiQuant <- 1 - ((rank(enfShort$seqclass_max_absdiff, ties.method = "average") - 1) / 
              (sum(!is.na(enfShort$seqclass_max_absdiff)) - 1))
enfShort$chr <- gsub(pattern = "chr", replacement = "", x = enfShort$chr)
enfShort <- enfShort[order(enfShort$chr, enfShort$pos, decreasing = FALSE),]
enfShort$chr <- paste0("chr",enfShort$chr)

In [34]:
##
##Extract scores for Loci hits
scoresSort1 <- data.frame(Variant = rep(c("rs12740374","rs398206"),6),
  Method = c(rep("CADD (phred)",2), rep("eQIF",2), rep("AVI (phred)",2), rep("LINSIGHT",2), rep("DVAR",2), rep("Sei",2)),
  Score = c(
    c(enfShort$caddPhred[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968], enfShort$caddPhred[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$eQIF[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$eQIF[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$aviPhred[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$aviPhred[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$LINSIGHT[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$LINSIGHT[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$DVAR[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$DVAR[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$seqclass_max_absdiff[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$seqclass_max_absdiff[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109])),
  Top = c(
    c(enfShort$caddQuant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$caddQuant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$eQIF_quant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$eQIF_quant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$aviQuant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$aviQuant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$linsightQuant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968], enfShort$linsightQuant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$dvarQuant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968],enfShort$dvarQuant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109]),
    c(enfShort$seiQuant[enfShort$chr == "chr1" & enfShort$ref == "G" & enfShort$alt == "T" & enfShort$pos == 109274968], enfShort$seiQuant[enfShort$chr == "chr21" & enfShort$ref == "C" & enfShort$alt == "A" & enfShort$pos == 41370109])))
write.table(scoresSort1, "../data/outputs/eval/scoresLoci.txt", append = FALSE, quote = FALSE, sep = "\t",row.names = FALSE, col.names = TRUE)

In [35]:
###
###
###
###
###
#Biological characterization
###
###
###
###
###

In [36]:
##
##Investigate the enrichment of the score for conserved sites (scaled)
enf$ifScale <- as.numeric(scale(enf$eQIF))
cols <- c(Primates = "phyloP447Primates",Mammals = "phyloP447",Vertebrates = "phyloPVertebrates")
estimates <- purrr::imap_dfr(cols, \(col, group) {
    x <- enf[[col]]
    y <- as.integer(x > quantile(x, 0.99, na.rm = TRUE))
mod <- glm(y ~ ifScale, family = binomial(), data = enf)
ci <- exp(confint(mod)[2, ])
tibble(OR = exp(coef(mod)[2]),CI_lower = ci[1],CI_upper = ci[2],Group = group)})

write.table(estimates,file = paste0("../data/outputs/eval/consEnr.txt"),append = FALSE,quote = FALSE,row.names = FALSE, col.names = TRUE)
estimates$Group <- factor(estimates$Group, levels = estimates$Group)
plotCons <- ggplot(estimates, aes(x = Group, y = OR)) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey50") +
  geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper),width = 0.15,linewidth = 0.6) +
  geom_point(size = 0.5) +
  labs(x = "Group", y = "Odds Ratio") +
  theme_classic()

Waiting for profiling to be done...

Waiting for profiling to be done...

Waiting for profiling to be done...



In [37]:
##
##Investigate the distribution of the score for gene regulatory annotations
enfRegCat <- data.frame(IFScore = c(enf$eQIF[enf$regReg == FALSE], 
                         enf$eQIF[enf$prom == TRUE],
                         enf$eQIF[enf$enh == TRUE],
                         enf$eQIF[enf$fp == TRUE & enf$prom == TRUE],
                         enf$eQIF[enf$fp == TRUE & enf$prom == TRUE & enf$phyloP447 > quantile(enf$phyloP447, probs = 0.99, na.rm = TRUE)],
                         enf$eQIF[enf$fp == TRUE & enf$enh == TRUE],
                         enf$eQIF[enf$fp == TRUE & enf$enh == TRUE & enf$phyloP447 > quantile(enf$phyloP447, probs = 0.99, na.rm = TRUE)]),
                       Group = c(rep("Not regulatory",nrow(enf[enf$regReg == FALSE,])), 
                                 rep("Promoter",nrow(enf[enf$prom == TRUE,])),
                                 rep("Enhancer",nrow(enf[enf$enh == TRUE,])),
                                 rep("Promoter FP",nrow(enf[enf$fp == TRUE & enf$prom == TRUE,])),
                                 rep("Promoter cons. FP",nrow(enf[enf$fp == TRUE & enf$prom == TRUE & enf$phyloP447 > quantile(enf$phyloP447, probs = 0.99, na.rm = TRUE),])),
                                 rep("Enhancer FP",nrow(enf[enf$fp == TRUE & enf$enh == TRUE,])),
                                 rep("Enhancer cons. FP",nrow(enf[enf$fp == TRUE & enf$enh == TRUE & enf$phyloP447 > quantile(enf$phyloP447, probs = 0.99, na.rm = TRUE),]))))
enfRegCat$Group <- reorder(enfRegCat$Group,enfRegCat$IFScore,FUN = median,na.rm = TRUE)
plotReg <- ggplot(enfRegCat, aes(x = Group, y = IFScore)) +geom_boxplot(outlier.shape = NA) + labs(x = "Group",y = "eQIF") +
  theme_classic()+ theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [38]:
##
##Write for AME
ameTop <- enf[order(-enf$eQIF, enf$id), ]
ameTop <- ameTop[1:5000, ]
ameTop$end <- ameTop$pos + 33
ameTop$pos <- ameTop$pos - 33
ameTop <- paste0(ameTop$chr, ":", ameTop$pos, "-", ameTop$end)
write.table(x = ameTop, file = "../data/outputs/eval/ame/regions.txt", col.names = FALSE, row.names = FALSE, append = FALSE, quote = FALSE, sep = "\t")

In [39]:
##
##run AME
system("./ameEnrichment.sh")

In [40]:
##
##Read AME and assign groups
ame <- read.delim("../data/outputs/eval/ame/ameOutJasparShuf/ame.tsv")
ame <- ame[ame$motif_ID != "",]
ame <- ame[ame$TP / ame$FP > 2,]

#Get motif gene names
x <- readLines("../data/raw/ame/jaspar.meme")
motifs <- grep("^MOTIF ", x, value = TRUE)
tf_names <- sub("^MOTIF\\s+\\S+\\s+", "", motifs)
tf_names <- sort(unique(tf_names))
tf_names <- unique(unlist(strsplit(tf_names, "::")))
tf_names <- toupper(sort(tf_names))
tfsCase <- ame$motif_alt_ID[ame$motif_alt_ID != ""]
tfsCase <- unique(unlist(strsplit(tfsCase, "::")))
tfsCase <- toupper(sort(tfsCase))

#Get similarity clusters
clusters <- read.delim("../data/raw/ame/clusters.tab")
for(i in 1:nrow(clusters)){
  clusters$id[i] <- paste(unlist(regmatches(clusters$id[i], gregexpr("MA[0-9]{4}\\.[0-9]+", clusters$id[i]))), collapse = ",")
}
clusters <- clusters[clusters$id != "",]
clusters <- clusters %>%
  separate_rows(id, sep = ",") %>%
  rename(ID = id)
colnames(ame)[3] <- "ID"
ame <- ame %>%
  left_join(clusters, by = "ID")

bestCluster <- ame %>%
  filter(!is.na(cluster)) %>%
  group_by(cluster) %>%
  slice_min(order_by = rank, n = 1, with_ties = FALSE) %>%
  ungroup()

# Write best motif per similarity cluster
write.table(x = bestCluster,file = "../data/outputs/eval/ame/ameOutJasparShuf/TF_Clusters_bestPerCluster.txt",
  append = FALSE,quote = FALSE,row.names = FALSE,col.names = TRUE,sep = "\t")

#Get biological classes
annotations <- fread("../data/raw/ame/annotation_table.txt", header = TRUE, data.table = FALSE, select = c(2,4))
colnames(annotations)[1] <- "ID"
ame <- ame %>%
  left_join(annotations, by = "ID")
#Make cluster plot
heatTab <- as.data.frame(table(ame$cluster, ame$class))

plotTfFamily <- ggplot(heatTab,
       aes(x = Var2,y = Var1,fill = Freq)) +
  geom_tile(color = "white") +
  scale_fill_gradient(low = "white", high = "darkred") +
  labs(x = "Biological cluster",y = "Similarity cluster",fill = "Number of TFs") +
  theme_minimal()+
theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
ggsave("../data/outputs/eval/ame/ameOutJasparShuf/TF_Clusters.pdf", plot = plotTfFamily, width = 7.5, height = 5)
ggsave("../data/outputs/eval/ame/ameOutJasparShuf/TF_Clusters.png", plot = plotTfFamily, width = 7.5, height = 5, dpi = 600)
write.table(x = ame, file = "../data/outputs/eval/ame/ameOutJasparShuf/TF_Clusters.txt", append = FALSE, quote = FALSE, row.names = FALSE, col.names = TRUE, sep = "\t")

In [48]:
plotCTCF <- plotMotifMutations(
    enf = enf,
    motif_path = "../data/raw/motifs/MA0139.2.meme",
    motif_length = 15,
    flank = 15,
    top_n = 5000)

plotETS1 <- plotMotifMutations(
    enf = enf,
    motif_path = "../data/raw/motifs/MA0098.4.meme",
    motif_length = 9,
    flank = 9,
    top_n = 5000)

Warning message in cor.test.default(apply(pfm_matrix, 2, max), mutation_counts$mutation_count, :
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  apply(pfm_matrix, 2, max) and mutation_counts$mutation_count
S = 167.24, p-value = 0.003573
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.7013523 



Warning message in cor.test.default(apply(pfm_matrix, 2, max), mutation_counts$mutation_count, :
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  apply(pfm_matrix, 2, max) and mutation_counts$mutation_count
S = 39.665, p-value = 0.04857
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.6694619 



In [49]:
plotBiol <- ((plotCons | plotReg) / (plotCTCF | plotETS1)) + plot_annotation(tag_levels = "A", tag_suffix = ")")
ggsave("../data/outputs/eval/plotBiol.pdf", plot = plotBiol, width = 7.5, height = 7.5)
ggsave("../data/outputs/eval/plotBiol.png", plot = plotBiol, width = 7.5, height = 7.5, dpi = 600)

In [43]:
###
###
###
###
###
#Write to output
###
###
###
###
###

In [44]:
##
##Write tables
write.table(enf, "../data/outputs/eval/enfHg38_all.txt", append = FALSE, quote = FALSE, sep = "\t",row.names = FALSE, col.names = TRUE)
write.table(enfShort[,colnames(enfShort) %in% c("chr","pos","ref","alt", "eQIF", "eQIF_quant")], "../data/outputs/eval/enfHg38_short_onlyeQIF.txt", append = FALSE, quote = FALSE, sep = "\t",row.names = FALSE, col.names = TRUE)
write.table(enfShort[,colnames(enfShort) %in% c("chr","pos","ref","alt", "eQIF","caddPhred", "LINSIGHT","DVAR", "seqclass_max_absdiff")], "../data/outputs/eval/enfHg38_short.txt", append = FALSE, quote = FALSE, sep = "\t",row.names = FALSE, col.names = TRUE)

In [3]:
#Read table
enf <- fread("../data/outputs/eval/enfHg38_all.txt",data.table = FALSE, header = TRUE)